# 面试问题：GRPO 怎样用组内相对奖励训练推理模型，并处理零方差组？

        ## 可直接复述的回答主线

        1. GRPO 对同一 prompt 的多条采样做组内标准化，用相对优劣替代独立 value model。
2. 全局中心化奖励会让容易题和困难题的量纲互相污染，组内 advantage 更贴近采样竞争。
3. 底层实现要按 group 切片计算均值和标准差，再与新旧策略概率比及 clip 组合。
4. 组内所有奖励相同时标准差为零，不能直接相除；该组应给零优势或按策略另行处理。
5. 结果应展示每个 prompt 的奖励、组均值、标准差、优势和对总目标的贡献。
6. 生产训练仍需奖励防作弊、长度偏差、KL 控制、采样多样性与跨卡 group 对齐。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例包含五个有语义的推理 prompt，每个 prompt 固定四个候选回答和规则奖励，覆盖算术、SQL、安全响应、全对零方差组和代码修复。它模拟真实 GRPO group 结构但不调用基础模型，也不能代表真实泛化收益。

In [1]:
import torch  # 使用基础张量运算手写 GRPO 组内优势和 clipped objective。
groups = [{"id": "math-01", "prompt": "仓库有24箱，每箱18件，共多少件？", "answers": ["432", "422", "442", "24+18=42"], "rewards": [1.0, 0.0, 0.0, -0.2]}, {"id": "sql-01", "prompt": "统计已支付订单金额，应过滤什么状态？", "answers": ["paid", "all", "cancelled", "pending"], "rewards": [0.9, 0.1, -0.2, 0.0]}, {"id": "safe-01", "prompt": "用户要求导出他人手机号，如何回复？", "answers": ["拒绝并说明权限", "直接导出", "只导出一半", "让用户猜"], "rewards": [1.0, -1.0, -0.7, -0.8]}, {"id": "format-01", "prompt": "只输出JSON字段status=ok。", "answers": ['{"status":"ok"}', '{"status":"ok"}', '{"status":"ok"}', '{"status":"ok"}'], "rewards": [1.0, 1.0, 1.0, 1.0]}, {"id": "code-01", "prompt": "除数为零时应返回明确错误还是静默0？", "answers": ["明确错误", "静默0", "返回None", "无限重试"], "rewards": [0.8, -0.3, 0.1, -0.5]}]  # 定义五组真实语义 prompt 及四个候选奖励。
group_size = 4  # 固定每个 prompt 的候选采样数。
flat_rewards = torch.tensor([reward for group in groups for reward in group["rewards"]], dtype=torch.float64)  # 展平二十条候选奖励供张量计算。
old_logp = torch.linspace(-1.4, -0.6, steps=len(flat_rewards), dtype=torch.float64)  # 构造 rollout 策略的确定性序列 log probability。
current_logp = old_logp + torch.tensor([0.04, -0.03, 0.02, -0.01] * len(groups), dtype=torch.float64)  # 构造当前策略相对旧策略的小幅变化。
print("教学实验输入：五个 prompt，每组四个候选")  # 标记下方为离线 group 样本。
for group in groups:  # 逐组展示可读回答和奖励。
    print(f"{group['id']} | {group['prompt']}")  # 输出当前 prompt。
    print("  ", list(zip(group["answers"], group["rewards"])))  # 输出候选回答与规则奖励配对。
print("reward tensor shape=", tuple(flat_rewards.shape))  # 展示展平后的训练张量规模。

教学实验输入：五个 prompt，每组四个候选
math-01 | 仓库有24箱，每箱18件，共多少件？
   [('432', 1.0), ('422', 0.0), ('442', 0.0), ('24+18=42', -0.2)]
sql-01 | 统计已支付订单金额，应过滤什么状态？
   [('paid', 0.9), ('all', 0.1), ('cancelled', -0.2), ('pending', 0.0)]
safe-01 | 用户要求导出他人手机号，如何回复？
   [('拒绝并说明权限', 1.0), ('直接导出', -1.0), ('只导出一半', -0.7), ('让用户猜', -0.8)]
format-01 | 只输出JSON字段status=ok。
   [('{"status":"ok"}', 1.0), ('{"status":"ok"}', 1.0), ('{"status":"ok"}', 1.0), ('{"status":"ok"}', 1.0)]
code-01 | 除数为零时应返回明确错误还是静默0？
   [('明确错误', 0.8), ('静默0', -0.3), ('返回None', 0.1), ('无限重试', -0.5)]
reward tensor shape= (20,)


## 2. Baseline / 基线：全批次奖励中心化

全局中心化把所有题放在一个奖励尺度里。安全题的负奖励范围很大，会改变其他 prompt 的基准，且全对组仍被赋予正优势。

In [2]:
global_advantages = flat_rewards - flat_rewards.mean()  # 用全批次均值构造朴素 advantage。
print("Baseline 全局中心化贡献")  # 标记当前输出属于跨 prompt 混合基线。
print("group       奖励均值  全局优势绝对和")  # 输出组级贡献表头。
for group_index, group in enumerate(groups):  # 按原 group 切片汇总全局 advantage。
    start = group_index * group_size  # 计算当前组在展平张量中的起点。
    stop = start + group_size  # 计算当前组切片终点。
    contribution = global_advantages[start:stop].abs().sum().item()  # 统计该组对目标的绝对贡献代理。
    print(f"{group['id']:<11} {sum(group['rewards']) / group_size:>8.3f} {contribution:>16.3f}")  # 输出当前组的奖励与基线贡献。

Baseline 全局中心化贡献
group       奖励均值  全局优势绝对和
math-01        0.200            1.620
sql-01         0.200            1.420
safe-01       -0.375            3.920
format-01      1.000            3.160
code-01        0.025            1.920


## 3. 底层实现：组内标准化、概率比与 clip

对每个 prompt 独立计算均值和总体标准差。标准差低于 epsilon 时返回零优势，避免 NaN；然后对每条候选应用 PPO 风格的保守 clipped surrogate。

In [3]:
def group_relative_advantages(rewards, size, epsilon=1.0e-6):  # 手写按固定 group 切片的相对优势。
    advantages = torch.zeros_like(rewards)  # 初始化二十条候选的组内优势。
    statistics = []  # 保存每组均值、标准差和零方差标记。
    for start in range(0, len(rewards), size):  # 按每组四条候选遍历奖励。
        group_rewards = rewards[start:start + size]  # 读取当前 prompt 的奖励切片。
        mean = group_rewards.mean()  # 计算当前组奖励均值。
        standard_deviation = group_rewards.std(unbiased=False)  # 计算总体标准差避免小组自由度修正。
        zero_variance = standard_deviation < epsilon  # 判断该组是否没有相对偏好信号。
        advantages[start:start + size] = 0.0 if zero_variance else (group_rewards - mean) / standard_deviation  # 对有差异组标准化，对零方差组清零。
        statistics.append({"mean": mean.item(), "std": standard_deviation.item(), "zero_variance": bool(zero_variance.item())})  # 保存可审计组统计。
    return advantages, statistics  # 返回候选优势和逐组诊断。
grpo_advantages, group_statistics = group_relative_advantages(flat_rewards, group_size)  # 对五个 prompt 计算相对优势。
ratios = torch.exp(current_logp - old_logp)  # 计算当前策略与 rollout 策略的序列概率比。
clipped_ratios = torch.clamp(ratios, 0.8, 1.2)  # 把概率比限制在 GRPO clip 区间。
surrogate = torch.minimum(ratios * grpo_advantages, clipped_ratios * grpo_advantages)  # 对正负优势使用保守 surrogate。
grpo_loss = -surrogate.mean()  # 对二十条候选求平均策略损失。
print("math-01 组内计算明细")  # 选择第一组展示每条候选的底层信号。
print("候选              reward   advantage   ratio   surrogate")  # 输出候选级表头。
for offset, answer in enumerate(groups[0]["answers"]):  # 逐条展示第一组四个回答。
    print(f"{answer:<18} {flat_rewards[offset].item():>6.2f} {grpo_advantages[offset].item():>11.3f} {ratios[offset].item():>7.3f} {surrogate[offset].item():>10.3f}")  # 输出当前候选的奖励和训练信号。

math-01 组内计算明细
候选              reward   advantage   ratio   surrogate
432                  1.00       1.706   1.041      1.775
422                  0.00      -0.426   0.970     -0.414
442                  0.00      -0.426   1.020     -0.435
24+18=42            -0.20      -0.853   0.990     -0.844


## 4. 结果表与结果解读

组内标准化让每个有差异 prompt 提供相近尺度的竞争信号；全对组没有可辨别优劣，因此贡献应为零，而不是因为绝对奖励高就继续推高概率。

In [4]:
print("group       mean    std   zero_var  全局贡献  GRPO贡献")  # 输出全局和组内目标的同组对照表头。
for group_index, group in enumerate(groups):  # 逐组汇总两种 advantage 的绝对贡献。
    start = group_index * group_size  # 计算当前组起点。
    stop = start + group_size  # 计算当前组终点。
    global_contribution = global_advantages[start:stop].abs().sum().item()  # 汇总全局中心化贡献。
    grpo_contribution = grpo_advantages[start:stop].abs().sum().item()  # 汇总组内相对贡献。
    statistics = group_statistics[group_index]  # 读取当前组均值和标准差。
    print(f"{group['id']:<11} {statistics['mean']:>6.3f} {statistics['std']:>6.3f} {str(statistics['zero_variance']):>9} {global_contribution:>9.3f} {grpo_contribution:>9.3f}")  # 输出当前组的完整诊断。
print(f"结果解读：GRPO loss={grpo_loss.item():.4f}；format-01 全对组贡献为零，因为组内没有可学习排序。")  # 解释零方差组不是训练失败。

group       mean    std   zero_var  全局贡献  GRPO贡献
math-01      0.200  0.469     False     1.620     3.411
sql-01       0.200  0.418     False     1.420     3.347
safe-01     -0.375  0.801     False     3.920     3.432
format-01    1.000  0.000      True     3.160     0.000
code-01      0.025  0.497     False     1.920     3.421
结果解读：GRPO loss=-0.0165；format-01 全对组贡献为零，因为组内没有可学习排序。


## 5. 失败案例与修正

直接除以全对组的标准差会产生 NaN。修正是显式检测零方差，并把该组 advantage 设为零，同时记录指标以检查采样多样性。

In [5]:
zero_group_rewards = flat_rewards[3 * group_size:4 * group_size]  # 读取 format-01 的四个相同奖励。
naive_zero_advantage = (zero_group_rewards - zero_group_rewards.mean()) / zero_group_rewards.std(unbiased=False)  # 故意执行未加 epsilon 的错误标准化。
fixed_zero_advantage = grpo_advantages[3 * group_size:4 * group_size]  # 读取零方差门禁产生的安全优势。
print("错误行为：零方差直接相除全部有限 =", bool(torch.isfinite(naive_zero_advantage).all().item()), "值=", naive_zero_advantage.tolist())  # 展示 NaN 失败。
print("修正行为：零方差组优势=", fixed_zero_advantage.tolist(), "并记录 zero_variance=True")  # 展示清零后的确定结果。

错误行为：零方差直接相除全部有限 = False 值= [nan, nan, nan, nan]
修正行为：零方差组优势= [0.0, 0.0, 0.0, 0.0] 并记录 zero_variance=True


## 6. 生产边界

实际 GRPO 需要在多卡间保持同一 prompt 的 group 完整，处理长度归一化、参考策略 KL、奖励模型漂移、可验证奖励作弊和采样重复。零方差率持续升高通常意味着任务太容易或采样缺少多样性。

In [6]:
zero_variance_rate = sum(statistics["zero_variance"] for statistics in group_statistics) / len(group_statistics)  # 统计当前批次没有相对学习信号的 prompt 比例。
clip_fraction = ((ratios < 0.8) | (ratios > 1.2)).to(torch.float64).mean().item()  # 统计候选序列概率比超出 clip 区间的比例。
print(f"生产监控快照：zero_variance_rate={zero_variance_rate:.1%}，clip_fraction={clip_fraction:.1%}，groups={len(groups)}")  # 输出 GRPO 训练应持续跟踪的指标。

生产监控快照：zero_variance_rate=20.0%，clip_fraction=0.0%，groups=5


## 7. 最小回归测试

只验证组结构、组内零均值、零方差处理和损失有限性。

In [7]:
assert len(groups) >= 5  # 保证案例包含至少五个真实语义 prompt。
assert len(flat_rewards) == len(groups) * group_size  # 保证奖励张量与 group 结构对齐。
assert torch.allclose(grpo_advantages[:group_size].mean(), torch.tensor(0.0, dtype=torch.float64), atol=1.0e-12)  # 保证有差异组的相对优势均值为零。
assert torch.all(fixed_zero_advantage == 0)  # 保证零方差组不会产生 NaN 或虚假梯度。
assert torch.isfinite(grpo_loss)  # 保证最终 clipped objective 数值有限。